In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import spherical_jn
from scipy.integrate import cumulative_trapezoid

csv_file = "/root/geant4/fac/Nb_all_wavefunctions.csv"
df = pd.read_csv(csv_file)
#===
d = df[df["Orbital"]=="Nb_4d32"].sort_values("r")

print(d[["r","P","Q"]].head(20))
print(d[["r","P","Q"]].tail(20))

print("max |P| =", np.max(np.abs(d["P"])))
print("max |Q| =", np.max(np.abs(d["Q"])))

print("norm P =", np.trapezoid(d["P"]**2, d["r"]))
print("norm Q =", np.trapezoid(d["Q"]**2, d["r"]))
print("fraction Q =", np.trapezoid(d["Q"]**2, d["r"]) /
      np.trapezoid(d["P"]**2 + d["Q"]**2, d["r"]))
# ===

df = df.dropna(subset=["r", "P", "Q"])

p = np.arange(0.0, 300.0 + 0.05, 0.05)
Qgrid = np.arange(0.0, 100.0 + 0.05, 0.05)

print(df["Orbital"].unique())


def get_orbital(df, name, print_values=True):
    d = df[df["Orbital"] == name].copy()
    d = d.sort_values("r")

    r_raw = d["r"].to_numpy()
    P_raw = d["P"].to_numpy()
    Q_raw = d["Q"].to_numpy()

    if print_values:
        print("\n======================")
        print(name)
        print("======================")
        print(d[["r", "P", "Q"]].head(8))
        print(d[["r", "P", "Q"]].tail(8))

    raw_norm = np.trapezoid(P_raw**2 + Q_raw**2, r_raw)
    print(name, "raw radial norm ∫(P²+Q²)dr =", raw_norm)

    # normalize raw FAC P,Q
    P_raw = P_raw / np.sqrt(raw_norm)
    Q_raw = Q_raw / np.sqrt(raw_norm)

    # interpolate to finer r grid
    r = np.linspace(r_raw.min(), r_raw.max(), 5000)
    P = np.interp(r, r_raw, P_raw)
    Qs = np.interp(r, r_raw, Q_raw)

    fine_norm = np.trapezoid(P**2 + Qs**2, r)
    P = P / np.sqrt(fine_norm)
    Qs = Qs / np.sqrt(fine_norm)

    print(name, "fine radial norm after interpolation =", np.trapezoid(P**2 + Qs**2, r))

    return r, P, Qs
   # return r_raw, P_raw, Q_raw

def chi_transform(p_grid, r, radial, l):
    chi = []

    for pp in p_grid:
        jl = spherical_jn(l, pp*r)

        # FAC P = rG, Q = rF
        # chi = sqrt(2/pi) int P(r) j_l(pr) r dr
        val = np.sqrt(2/np.pi) * np.trapezoid(radial * jl * r, r)

        chi.append(val)

    return np.array(chi)


def calculate_J(p_grid, chiG, chiF, Qgrid):
    integrand = (chiG**2 + chiF**2) * p_grid

    rev = cumulative_trapezoid(
        integrand[::-1],
        p_grid[::-1],
        initial=0
    )

    J_p = -0.5 * rev[::-1]
    J_Q = np.interp(Qgrid, p_grid, J_p)

    return J_Q


def one_orbital_J(df, name, l_large, l_small):
    r, P, Qs = get_orbital(df, name)

    chiG = chi_transform(p, r, P, l_large)
    chiF = chi_transform(p, r, Qs, l_small)
    # ====
    print("Integral chiG^2 dp =", np.trapezoid(chiG**2, p))
    print("Integral chiF^2 dp =", np.trapezoid(chiF**2, p))
    print("Integral I(p) dp =", np.trapezoid((chiG**2+chiF**2)*p**2, p))
    # ===
    I = (chiG**2 + chiF**2) * p**2
    print(name, "momentum norm ∫I(p)dp =", np.trapezoid(I, p))

    J = calculate_J(p, chiG, chiF, Qgrid)
    print(name, "Compton check 2∫J(Q)dQ =", 2*np.trapezoid(J, Qgrid))

    return J


# =====================================================
# Relativistic orbitals
# large component l = orbital l
# small component l' is decided by j
# =====================================================

J_1s12 = one_orbital_J(df, "Nb_1s12", 0, 1)

J_2s12 = one_orbital_J(df, "Nb_2s12", 0, 1)
J_2p12 = one_orbital_J(df, "Nb_2p12", 1, 0)
J_2p32 = one_orbital_J(df, "Nb_2p32", 1, 2)

J_3s12 = one_orbital_J(df, "Nb_3s12", 0, 1)
J_3p12 = one_orbital_J(df, "Nb_3p12", 1, 0)
J_3p32 = one_orbital_J(df, "Nb_3p32", 1, 2)
J_3d32 = one_orbital_J(df, "Nb_3d32", 2, 1)
J_3d52 = one_orbital_J(df, "Nb_3d52", 2, 3)

J_4s12 = one_orbital_J(df, "Nb_4s12", 0, 1)
J_4p12 = one_orbital_J(df, "Nb_4p12", 1, 0)
J_4p32 = one_orbital_J(df, "Nb_4p32", 1, 2)
J_4d32 = one_orbital_J(df, "Nb_4d32", 2, 1)
J_4d52 = one_orbital_J(df, "Nb_4d52", 2, 3)

J_5s12 = one_orbital_J(df, "Nb_5s12", 0, 1)

J_total_Nb_rel = (
    2*J_1s12
    + 2*J_2s12
    + 2*J_2p12 + 4*J_2p32
    + 2*J_3s12
    + 2*J_3p12 + 4*J_3p32
    + 4*J_3d32 + 6*J_3d52
    + 2*J_4s12
    + 2*J_4p12 + 4*J_4p32
    + 4*J_4d32 + 0*J_4d52
    + 1*J_5s12
)

print("\nTotal Nb check 2∫Jtotal dQ =", 2*np.trapezoid(J_total_Nb_rel, Qgrid))
print("Expected close to 41")


table = pd.DataFrame({
    "Q": Qgrid,
    "J_1s1/2(2)": 2*J_1s12,
    "J_2s1/2(2)": 2*J_2s12,
    "J_2p1/2(2)": 2*J_2p12,
    "J_2p3/2(4)": 4*J_2p32,
    "J_3s1/2(2)": 2*J_3s12,
    "J_3p1/2(2)": 2*J_3p12,
    "J_3p3/2(4)": 4*J_3p32,
    "J_3d3/2(4)": 4*J_3d32,
    "J_3d5/2(6)": 6*J_3d52,
    "J_4s1/2(2)": 2*J_4s12,
    "J_4p1/2(2)": 2*J_4p12,
    "J_4p3/2(4)": 4*J_4p32,
    "J_4d3/2(4)": 4*J_4d32,
    "J_4d5/2(0)": 0*J_4d52,
    "J_5s1/2(1)": 1*J_5s12,
    "J_total_Nb_relativistic": J_total_Nb_rel
})

table.to_csv("Niobium_relativistic_Compton_profile.csv", index=False)
print(table)


plt.figure(figsize=(8,6))
plt.plot(Qgrid, 2*J_1s12, label="1s1/2(2)")
plt.plot(Qgrid, 2*J_2s12, label="2s1/2(2)")
plt.plot(Qgrid, 2*J_2p12 + 4*J_2p32, label="2p(6)")
plt.plot(Qgrid, 2*J_3s12, label="3s1/2(2)")
plt.plot(Qgrid, 2*J_3p12 + 4*J_3p32, label="3p(6)")
plt.plot(Qgrid, 4*J_3d32 + 6*J_3d52, label="3d(10)")
plt.plot(Qgrid, 2*J_4s12, label="4s1/2(2)")
plt.plot(Qgrid, 2*J_4p12 + 4*J_4p32, label="4p(6)")
plt.plot(Qgrid, 4*J_4d32 + 0*J_4d52, label="4d(4)")
plt.plot(Qgrid, 1*J_5s12, label="5s(1)")
plt.plot(Qgrid, J_total_Nb_rel, label="Nb total relativistic", linewidth=2)

plt.xlabel("Q (a.u.)")
plt.ylabel("J(Q)")
plt.title("Relativistic Niobium Compton Profile")
plt.yscale("log")
plt.grid(True)
plt.legend()
plt.show()

                 r             P             Q
2222  2.439024e-08  1.159748e-19  4.996821e-15
2223  2.674584e-08  1.525340e-19  6.008490e-15
2224  2.932889e-08  2.008074e-19  7.225183e-15
2225  3.216141e-08  2.645161e-19  8.688152e-15
2226  3.526742e-08  3.485682e-19  1.044734e-14
2227  3.867340e-08  4.594374e-19  1.256270e-14
2228  4.240824e-08  6.056605e-19  1.510637e-14
2229  4.650376e-08  7.984955e-19  1.816489e-14
2230  5.099481e-08  1.052788e-18  2.184290e-14
2231  5.591938e-08  1.388109e-18  2.626560e-14
2232  6.131951e-08  1.830273e-18  3.158328e-14
2233  6.724114e-08  2.413312e-18  3.797773e-14
2234  7.373449e-08  3.182097e-18  4.566697e-14
2235  8.085475e-08  4.195797e-18  5.491289e-14
2236  8.866243e-08  5.532422e-18  6.603019e-14
2237  9.722405e-08  7.294840e-18  7.939795e-14
2238  1.066122e-07  9.618671e-18  9.547232e-14
2239  1.169068e-07  1.268273e-17  1.148006e-13
2240  1.281951e-07  1.672279e-17  1.380406e-13
2241  1.405735e-07  2.204972e-17  1.659854e-13
             

PermissionError: [Errno 13] Permission denied: 'Niobium_relativistic_Compton_profile.csv'